In [ ]:
import random
import time
import numpy as np
from collections import defaultdict

In [ ]:
# PACKETIZE DA DATA
# packs sequences together into a fixed length and pads the end

folder = '/home/user/Desktop/ml/gpt_dt/datasets/instruction_fine_tuning'
split = 'train'

pad_token_id = 30000
eot_id = 0
target_block_size = 1024 # the block size that you are packing the data into

input_file =f'{folder}/{split}.bin'
output_file = f'{folder}/packed_{split}_{target_block_size}.bin'

def process_efficiently():
    start_time = time.time()
    
    # Map input (Uses ~0 RAM)
    all_data = np.memmap(input_file, dtype=np.uint16, mode='r')

    
    eot_indices = []
    chunk_size = 1_000_000

    for start in range(0, len(all_data), chunk_size):
        chunk = all_data[start:start + chunk_size]

        indices = np.flatnonzero(chunk == eot_id) + start
        eot_indices.extend(indices.tolist())

    if len(eot_indices) == 0:
        raise ValueError("No EOT tokens found!")

    if eot_indices[-1] != len(all_data) - 1:
        raise ValueError("Dataset contains trailing tokens after final EOT!")

    after_eot_indices = np.array(eot_indices, dtype=np.int64) + 1

    seq_start_indices = np.empty_like(after_eot_indices)
    seq_start_indices[0] = 0
    seq_start_indices[1:] = after_eot_indices[:-1]

    lengths = after_eot_indices - seq_start_indices

    # Create structured metadata array
    metadata = np.zeros(
        len(seq_start_indices),
        dtype=[('ptr', 'u8'), ('length', 'u2')]
    )

    if np.any(lengths > target_block_size):
        raise ValueError(
            f"Found {np.sum(lengths > target_block_size)} "
            f"sequences exceeding {target_block_size} tokens."
        )

    # Fill fields
    metadata['ptr'] = seq_start_indices
    metadata['length'] = lengths

    metadata = metadata[metadata['length'] > 0]

    # SORT: Biggest sequences first (The "D" in FFD)
    print("Sorting metadata by length...")
    metadata.sort(order='length')
    metadata = metadata[::-1] # Reverse for descending

    # PACK: First Fit logic with Capacity Lookup
    print(f"Packing {len(metadata)} items into {target_block_size} bins...")
    
    # bin_contents will store indices of metadata for each 1024-block
    bins = []
    bin_spaces = [] # To track remaining space in each bin
    space_buckets = defaultdict(list)

    for ptr, length in metadata:
        found_bin_idx = -1
        
        # Look for a bin that has AT LEAST 'length' space left
        # We check from 'length' up to 1024
        for space_available in range(length, target_block_size + 1):
            if space_buckets[space_available]:
                found_bin_idx = space_buckets[space_available].pop()
                break
        
        if found_bin_idx != -1:
            # Found a fit! Put it in the bin
            bins[found_bin_idx].append((ptr, length))
            bin_spaces[found_bin_idx] += length
            
            # Re-calculate remaining space and put back in a new bucket
            new_remaining = target_block_size - bin_spaces[found_bin_idx]
            if new_remaining > 0:
                space_buckets[new_remaining].append(found_bin_idx)
        else:
            # No existing bin has enough space, create a new one
            new_idx = len(bins)
            bins.append([(ptr, length)])
            bin_spaces.append(length)
            
            # Calculate remaining space and put in a bucket
            remaining = target_block_size - length
            if remaining > 0:
                space_buckets[remaining].append(new_idx)

    # WRITE: Transfer tokens directly to output memmap
    print(f"Writing {len(bins)} packed blocks to disk...")
    out_data = np.memmap(output_file, dtype=np.uint16, mode='w+', shape=(len(bins) * target_block_size + 1,))
    
    random.shuffle(bins)

    out_ptr = 0
    for b in bins:
        current_fill = 0
        for ptr, length in b:
            p, l = int(ptr), int(length)
            out_data[out_ptr : out_ptr + l] = all_data[p : p + l]
            out_ptr += l
            current_fill += l
        
        # Fill remaining gap with padding
        gap = target_block_size - current_fill
        if gap > 0:
            out_data[out_ptr : out_ptr + gap] = pad_token_id
            out_ptr += gap
    out_data[out_ptr] = pad_token_id # append one last pad to make it eASier for the dataloader which grabs batch+1 for the y

    out_data.flush()
    print(f"Finished in {time.time() - start_time:.2f} seconds.")

process_efficiently()